# Carga y reporte de coachtwin-workouts

En este notebook se construye un flujo completo de trabajo:

1. Configurar la ruta del proyecto para importar las clases reutilizables.
2. Cargar `coachtwin-workouts` desde Hugging Face.
3. Obtener un `DataFrame` de pandas y revisar su estructura.
4. Generar un reporte en pantalla y guardarlo en un archivo `.txt`.

## 1. Preparar el entorno e importar la clase de carga

El notebook puede ejecutarse desde distintas carpetas. Por eso se buscan los directorios padres hasta encontrar `src/clases_po` y se agrega `src` a `sys.path`. Así se puede importar `HFDatasetLoader` sin instalar el proyecto como paquete.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import HFDatasetLoader

/home/runner/work/Proyecto-Integrado-III/Proyecto-Integrado-III/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configurar el dataset y la salida

`dataset_name` identifica el repositorio en Hugging Face y `split="train"` indica la división que se descargará. El parámetro `filename` hace que el `DataFrame` también se guarde como CSV en `src/sincronicas/datos/` cuando el notebook se ejecuta desde esa carpeta.

In [2]:
loader = HFDatasetLoader(
    dataset_name="OrDora/coachtwin-workouts",
    split="train",
    filename="datos/coachtwin-workouts.csv",
)


## 3. Descargar el dataset y crear el DataFrame

El método `load()` descarga el dataset, lo convierte a `pandas.DataFrame` y lo asigna a la variable `df`. La expresión `df.head()` muestra las primeras filas para comprobar que la carga fue exitosa.

In [3]:
df = loader.load()
df.head()

Generating train split:   0%|          | 0/10393 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 10393/10393 [00:00<00:00, 198656.53 examples/s]

,title,warm_up,exercises,cool_down,notes,goal,equipment,duration_minutes,difficulty,body_focus,source
0,Intermediate Leg Strength & Endurance Workout,6 minutes of dynamic stretching focusing on qu...,"[{'name': 'Deadlifts', 'sets': 3, 'reps': '9-1...",5 minutes of static stretching focusing on qua...,Rest properly between sets to avoid injury. In...,weight loss,no equipment (bodyweight only),30,intermediate,legs,v1_prompt
1,Strength Training Circuit Legs,"5 minutes of dynamic stretches, involving leg ...","[{'name': 'Leg press (2 sets, 8 reps, rest 90 ...","5 minutes of static stretching, focusing on ha...",Maintain proper form during exercises and ensu...,flexibility,no equipment (bodyweight only),15,beginner,legs,v1_prompt
2,Upper Body Workout Session,5 minutes of dynamic stretches followed by a l...,"[{'name': 'Bicep Curls', 'sets': 6, 'reps': '3...",5 minutes of static stretching focusing on bac...,"Include rest of the day off from workouts, ens...",flexibility,resistance bands,45,intermediate,upper body,v1_prompt
3,Legs Full Strength Circuit,3 minutes of jogging at a moderate pace,"[{'name': 'Seated Leg Presses', 'sets': 3, 're...",5 minutes of dynamic stretches focusing on cor...,Ensure proper alignment and technique during e...,strength,no equipment (bodyweight only),60,intermediate,legs,v1_prompt
4,High Intensity Interval Training (HIIT) Cardio...,15 minutes of brisk walking or jogging at a mo...,"[{'name': 'Bent Over Rows', 'sets': 3, 'reps':...",5 minutes of dynamic stretching focused on qua...,Rest for 1-minute before each exercise. Mix up...,strength,dumbbells,45,intermediate,full body,v1_prompt


## 4. Aplicar transformaciones mínimas al DataFrame

En este paso realizamos **cuatro transformaciones sencillas** sobre el DataFrame cargado desde Hugging Face.  
El objetivo es mostrar cómo podemos modificar y preparar los datos antes de generar el reporte final:

1. **Renombrar columnas** → cambiamos `title` por `workout_title` para mayor claridad.  
2. **Filtrar filas** → seleccionamos únicamente rutinas con duración mayor a 20 minutos.  
3. **Crear columna derivada** → añadimos `duration_hours` calculada a partir de `duration_minutes`.  
4. **Seleccionar columnas relevantes** → dejamos solo las columnas necesarias para el análisis:  
   `workout_title`, `duration_minutes`, `duration_hours`, `difficulty`, `body_focus`.

Estas transformaciones garantizan que el reporte se genere sobre un DataFrame más limpio, claro y enfocado en la información relevante.


In [4]:
# 1. Renombrar una columna para mayor claridad
df = df.rename(columns={"title": "workout_title"})

# 2. Filtrar filas con duración mayor a 20 minutos
df = df[df["duration_minutes"] > 20]

# 3. Crear una nueva columna derivada (horas de duración)
df["duration_hours"] = df["duration_minutes"] / 60

# 4. Seleccionar solo columnas relevantes para el análisis
df = df[["workout_title", "duration_minutes", "duration_hours", "difficulty", "body_focus"]]

df.head()


,workout_title,duration_minutes,duration_hours,difficulty,body_focus
0,Intermediate Leg Strength & Endurance Workout,30,0.50,intermediate,legs
2,Upper Body Workout Session,45,0.75,intermediate,upper body
3,Legs Full Strength Circuit,60,1.00,intermediate,legs
4,High Intensity Interval Training (HIIT) Cardio...,45,0.75,intermediate,full body
6,Max Strength Training Circuit,60,1.00,advanced,full body


## 5. Revisar la estructura del DataFrame

Esta celda realiza una revisión inicial: cuenta filas y columnas y muestra el tipo de cada columna junto con información de memoria y valores no nulos. Esta inspección ayuda a detectar problemas antes de analizar los datos.

In [5]:
print(f"Filas: {len(df)}")
print(f"Columnas: {len(df.columns)}")
df.info()

Filas: 7886
Columnas: 5
<class 'pandas.core.frame.DataFrame'>
Index: 7886 entries, 0 to 10390
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   workout_title     7886 non-null   object 
 1   duration_minutes  7886 non-null   int64  
 2   duration_hours    7886 non-null   float64
 3   difficulty        7886 non-null   object 
 4   body_focus        7886 non-null   object 
dtypes: float64(1), int64(1), object(3)
memory usage: 369.7+ KB


## 6. Importar la clase de reportes

`Reportes` recibe un `DataFrame` en sus métodos y permite generar un resumen textual, imprimirlo en la consola o guardarlo en un archivo `.txt`.

In [6]:
from clases_po import Reportes

## 7. Generar y visualizar el reporte como texto

`generar_reporte(df)` devuelve un `str` con el número de filas, columnas, duplicados, tipos de datos, valores nulos y estadísticas numéricas. En esta celda el texto se guarda temporalmente en `texto` y se imprime.

In [7]:
reportes = Reportes()
texto = reportes.generar_reporte(df)

print(texto)

REPORTE DEL DATAFRAME
Filas: 7886
Columnas: 5
Duplicados: 863

Columnas y tipos:
- workout_title: object
- duration_minutes: int64
- duration_hours: float64
- difficulty: object
- body_focus: object

Valores nulos por columna:
- workout_title: 0
- duration_minutes: 0
- duration_hours: 0
- difficulty: 0
- body_focus: 0

Estadísticas numéricas:
       duration_minutes  duration_hours
count          7886.000        7886.000
mean             45.308           0.755
std              12.205           0.203
min              30.000           0.500
25%              30.000           0.500
50%              45.000           0.750
75%              60.000           1.000
max              60.000           1.000

Mediana de columnas numéricas:
duration_minutes    45.00
duration_hours       0.75
dtype: float64

Valores únicos por columna categórica:
- workout_title: 5120
- difficulty: 3
- body_focus: 8

Distribución de difficulty:
difficulty
advanced        2886
intermediate    2548
beginner        2452

## 8. Imprimir el reporte directamente

`imprimir_reporte(df)` ejecuta la generación y muestra el resultado directamente. Es útil para consultar rápidamente el resumen durante una ejecución interactiva.

In [8]:
reportes.imprimir_reporte(df)

REPORTE DEL DATAFRAME
Filas: 7886
Columnas: 5
Duplicados: 863

Columnas y tipos:
- workout_title: object
- duration_minutes: int64
- duration_hours: float64
- difficulty: object
- body_focus: object

Valores nulos por columna:
- workout_title: 0
- duration_minutes: 0
- duration_hours: 0
- difficulty: 0
- body_focus: 0

Estadísticas numéricas:
       duration_minutes  duration_hours
count          7886.000        7886.000
mean             45.308           0.755
std              12.205           0.203
min              30.000           0.500
25%              30.000           0.500
50%              45.000           0.750
75%              60.000           1.000
max              60.000           1.000

Mediana de columnas numéricas:
duration_minutes    45.00
duration_hours       0.75
dtype: float64

Valores únicos por columna categórica:
- workout_title: 5120
- difficulty: 3
- body_focus: 8

Distribución de difficulty:
difficulty
advanced        2886
intermediate    2548
beginner        2452

## 9. Guardar el reporte en un archivo de texto

`guardar_reporte(df, ruta)` crea los directorios necesarios y escribe el reporte en UTF-8. El archivo quedará en `src/sincronicas/reportes/coachtwin-workouts.txt` cuando se ejecute desde `src/sincronicas`.

In [9]:
ruta = reportes.guardar_reporte(
    df,
    "reportes/coachtwin-workouts.txt",
)

print(f"Reporte guardado en: {ruta}")

Reporte guardado en: reportes/coachtwin-workouts.txt
